# 01 - Explore the data and build scikit-learn baselines

Three questions, one rule: **always measure on matches the model has never seen.**

1. **Draft** - can you predict the winner from what you know before the game (champion, lane, side)?
2. **Post-game** - can you predict the winner from a player's final stats?
3. **By minute** - how well can you predict the winner *during* the game, minute by minute?

This notebook runs on a laptop in about two minutes. Set `LOLML_SMOKE=1` to use the small sample that ships with the repo.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make the `lolml` helpers importable (notebooks run from notebooks/)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from lolml import data, features as F, paths
paths.ensure_dirs()
SMOKE = bool(os.environ.get("LOLML_SMOKE"))   # LOLML_SMOKE=1 uses the small committed sample and short training budgets
SEED = 42
def save_fig(name):
    plt.tight_layout(); plt.savefig(paths.OUT / name, dpi=110); plt.show()


## 1. The post-game dataset

One row per player per finished ranked match (10 rows per match). Player names and IDs were removed when the dataset was built.

In [ ]:
perf = data.load("performance")
print(f"{len(perf):,} rows, {perf['match_id'].nunique():,} matches, {perf.shape[1]} columns")
perf[["match_id", "championName", "teamPosition", "teamId", "kills", "deaths", "assists", "goldEarned", "duration", "win"]].head(10)

In [ ]:
print("share of winning rows:", round(perf["win"].mean(), 3))
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
perf["duration"].clip(upper=60).hist(bins=40, ax=ax[0]); ax[0].set_title("game length (minutes)")
perf["deaths"].clip(upper=20).hist(bins=20, ax=ax[1]); ax[1].set_title("deaths per player")
perf.groupby("teamPosition")["win"].mean().plot.bar(ax=ax[2], ylim=(0.4, 0.6)); ax[2].set_title("win rate by position")
save_fig("explore.png")

## 2. Split by match, not by row

All 10 players of a match share the same game. If one lands in the training set and another in the test set, the model gets to "see" the match it is tested on. `group_split` keeps every match on one side.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
train_idx, test_idx = F.group_split(perf, test_size=0.2, seed=SEED)
train, test = perf.iloc[train_idx], perf.iloc[test_idx]
assert set(train["match_id"]).isdisjoint(test["match_id"])
print(f"train {len(train):,} rows / {train['match_id'].nunique():,} matches   test {len(test):,} rows / {test['match_id'].nunique():,} matches")

results = {}
def report(name, model, cols, tr, te, target="win"):
    model.fit(tr[cols], tr[target])
    p = model.predict_proba(te[cols])[:, 1]
    acc, auc = accuracy_score(te[target], p > 0.5), roc_auc_score(te[target], p)
    results[name] = {"accuracy": round(acc, 4), "auc": round(auc, 4), "rows_test": len(te)}
    print(f"{name:<42} accuracy {acc:.3f}   AUC {auc:.3f}")
    return model

## 3. Question 1 - draft only

Before the game you know the champion you picked, your lane, your side of the map and your summoner spells. A logistic regression with one-hot encoded champions is the classic baseline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

draft_cols = F.DRAFT_CAT + F.DRAFT_NUM
draft_lr = make_pipeline(
    ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), F.DRAFT_CAT + ["teamId", "summoner1Id", "summoner2Id"])]),
    LogisticRegression(max_iter=1000))
report("Q1 draft only (logistic regression)", draft_lr, draft_cols, train, test);

About 50 % is a coin flip. Picking the champion does not decide a ranked game on its own, and the data agrees. Anything you gain before minute 1 has to come from information this dataset does not contain (rank, mastery, your teammates, the ban phase).

## 4. Question 2 - post-game player stats

Now use everything a player did: kills, deaths, assists, gold, damage, vision, minions. `post_game_numeric` drops the columns that only *describe how the game ended* (nexus, inhibitors, surrender flags). Notebook 04 shows what happens if you keep them.

In [ ]:
post_cols = F.post_game_numeric(perf)
print(len(post_cols), "post-game features, for example:", post_cols[:8])
post_lr = make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=2000))
report("Q2 post-game (logistic regression)", post_lr, post_cols, train, test)
post_hgb = report("Q2 post-game (HistGradientBoosting)", HistGradientBoostingClassifier(max_iter=100 if SMOKE else 300, random_state=SEED), post_cols, train, test)

A simple linear model already gets most of the way; gradient boosting adds a little. This model **explains** a finished game. It cannot help you in champion select or in minute 5.

### Which stats does it use?

Permutation importance shuffles one column at a time on the *test* set and measures how much the AUC drops. A large drop means the model relies on that column.

In [ ]:
from sklearn.inspection import permutation_importance
sub = test.sample(min(len(test), 15000), random_state=SEED)
pi = permutation_importance(post_hgb, sub[post_cols], sub["win"], scoring="roc_auc", n_repeats=3, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=post_cols).sort_values(ascending=False)
imp.head(12)[::-1].plot.barh(figsize=(6, 4), title="Permutation importance (drop in AUC)")
save_fig("perm_importance_postgame.png")
imp.head(10).round(4)

## 5. Question 3 - by minute

The second dataset has one row per player per snapshot *during* a match: 18 stats such as health, armor and attack damage, taken from the same match timeline that the Live Client API mirrors. There are no kills or gold in it. The target is whether that player's team goes on to win.

In [ ]:
bym = data.load("by_minute")
print(f"{len(bym):,} rows, {bym['match_id'].nunique():,} matches, minutes {bym['minute'].min():.1f} to {bym['minute'].max():.1f}")
bm_train_idx, bm_test_idx = F.group_split(bym, test_size=0.2, seed=SEED)
bm_train, bm_test = bym.iloc[bm_train_idx], bym.iloc[bm_test_idx]
by_all = report("Q3 by minute (all 18 stats)", HistGradientBoostingClassifier(max_iter=100 if SMOKE else 300, random_state=SEED), F.BYMIN_ALL, bm_train, bm_test, target="winner")
by_live = report("Q3 by minute (live-safe stats only)", HistGradientBoostingClassifier(max_iter=100 if SMOKE else 300, random_state=SEED), F.BYMIN_LIVE_SAFE, bm_train, bm_test, target="winner")

In [ ]:
p = by_live.predict_proba(bm_test[F.BYMIN_LIVE_SAFE])[:, 1]
correct = pd.Series((p > 0.5) == bm_test["winner"].values, index=bm_test.index)
abm = F.accuracy_by_minute(bm_test["minute"], correct)
print(abm.round(3).T.to_string())
ax = abm["accuracy"].plot(marker="o", figsize=(7, 3.5), ylim=(0.45, 0.75), title="Accuracy by game minute (unseen matches)")
ax.axhline(0.5, color="grey", ls="--"); ax.set_ylabel("accuracy")
save_fig("accuracy_by_minute.png")

At the start of the game the model is guessing (about 50 %). Around minute 20 it is right about two times in three, even though it only sees one player's 18 stats. Nobody can call a League game at minute 3 from stats like these, and the numbers make that visible.

## 6. The honest scoreboard

In [ ]:
table = pd.DataFrame(results).T
table["question"] = ["draft", "post-game", "post-game", "by minute", "by minute"][: len(table)]
print(table.to_string())
(paths.OUT / "baseline_results.json").write_text(json.dumps(results, indent=2))
fig, ax = plt.subplots(figsize=(8, 3.5))
best = {"Q1 draft": results["Q1 draft only (logistic regression)"]["accuracy"],
        "Q3 by minute\n(live-safe stats)": results["Q3 by minute (live-safe stats only)"]["accuracy"],
        "Q2 post-game": results["Q2 post-game (HistGradientBoosting)"]["accuracy"]}
ax.bar(best.keys(), best.values(), color=["#c74634", "#f0cc71", "#2d5967"]); ax.axhline(0.5, color="grey", ls="--")
ax.set_ylim(0.4, 1.0); ax.set_ylabel("accuracy on unseen matches")
for i, v in enumerate(best.values()): ax.text(i, v + 0.01, f"{v:.1%}", ha="center")
save_fig("three_questions.png")